# 03 — Random forest: diagnosis and tuning (Manel)

Development set only (5-fold CV); the test set is not used.

**Scripts run, this notebook explains.** Every number here is produced by
`tuning/random_forest.py` and saved in `tuning/results/random_forest/`; this
notebook only reads those files and comments on them.

| Step | Command (from the project root) | Output |
|---|---|---|
| 1-2. Baseline + diagnosis | `python -m tuning.random_forest baseline` | `baseline.csv` |
| 3. Validation curves | `python -m tuning.random_forest curves` | `curves.csv`, `curves/*.png` |
| 4. Tuning v2 | `python -m tuning.tune --version v2 --model rf` | `tuning/results/v2/best_params.json` |

Tuning v1 = Soumaya's first Optuna run: `tuning/results/v1/best_params.json` (git tag `baseline-v1`).

Hyperparameters studied: `max_depth`, `min_samples_leaf`, `min_samples_split`, `max_features`, `max_samples`, `criterion`, `n_estimators`, `class_weight`.

In [4]:
import sys
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))
RESULTS = ROOT / "tuning" / "results" / "random_forest"

## 1. Baseline

**How a random forest "minimises the loss".** Unlike logistic regression or
LightGBM, a random forest does not minimise one global log-loss. Each tree
chooses, at every node, the split that reduces **impurity** the most (Gini by
default: how mixed the two classes are in a node). A fully grown tree keeps
splitting until its leaves are pure, so it memorises the training data. The
forest averages many trees, each trained on a bootstrap sample and a random
subset of features, which reduces that variance.

So most RF hyperparameters control **how far each tree may grow**
(`max_depth`, `min_samples_leaf`, `min_samples_split`) or **how different the
trees are** (`max_features`, `max_samples`).

Five starting points, from the floor to the tuned v1 model (5-fold CV):

In [5]:
baseline = pd.read_csv(RESULTS / "baseline.csv")
baseline[["experiment", "train PR-AUC", "val PR-AUC", "val PR-AUC std", "gap PR-AUC",
          "val ROC-AUC", "val Brier"]].round(3)

,experiment,train PR-AUC,val PR-AUC,val PR-AUC std,gap PR-AUC,val ROC-AUC,val Brier
0,Dummy (predicts the base rate),0.239,0.239,0.000,-0.000,0.500,0.182
1,"One decision tree, fully grown",0.977,0.444,0.010,0.533,0.734,0.196
2,"RF sklearn defaults (100 trees, fully grown)",0.972,0.639,0.011,0.333,0.857,0.131
3,"RF baseline (300 trees, min_samples_leaf 5)",0.802,0.711,0.012,0.092,0.890,0.111
4,RF tuned v1,0.749,0.715,0.013,0.035,0.891,0.136


PR-AUC: higher is better; 0.239 is the floor (the share of high earners).
Brier: quality of the probabilities, lower is better.

## 2. Diagnosis: train vs validation (overfitting or underfitting?)

1. **One fully grown tree memorises:** 0.977 on the data it learned from,
   0.444 on new data (gap 0.53). High variance.
2. **A forest averages part of that noise away, not all of it:** sklearn
   defaults reach 0.639, but the trees are still fully grown and the gap is
   still 0.33.
3. **Bigger leaves fix the overfitting:** at least 5 people per leaf
   (`min_samples_leaf=5`) shrinks the gap to 0.09 and lifts validation to
   0.711. The parameters that limit tree growth are the ones that matter most.
4. **Tuning v1 adds nothing measurable:** 0.711 → 0.715, while the fold-to-fold
   noise is ± 0.013. Its gap (0.035) is small: overfitting is no longer the problem.
5. **Tuning v1 made the probabilities worse:** Brier 0.111 → 0.136, because of
   `class_weight="balanced_subsample"`, which pushes every probability up. The
   ranking is unchanged (ROC-AUC 0.890 vs 0.891), but the probabilities can no
   longer be read as real chances, which matters for calibration and fairness.
6. **The ceiling is the data, not the settings:** LR 0.708, RF 0.715 and
   LightGBM 0.716 all stop around 0.71 PR-AUC with 6 inputs. Expect tuning to
   move results by about ± 0.01.

**Questions for the validation curves:** where does the gap close without
validation dropping (`min_samples_leaf`, `max_depth`, `min_samples_split`)?
Does decorrelating the trees help (`max_features`, `max_samples`)? Does class
weighting cost calibration when tested alone (`class_weight`, Brier)?

## 3. Validation curves on the key hyperparameters

One hyperparameter changes, the others stay at the sklearn defaults with 200
trees. Run `python -m tuning.random_forest curves` (all curves) or
`python -m tuning.random_forest curves --param max_depth` (one curve), then
re-run the cell below.

In [6]:
path = RESULTS / "curves.csv"
if path.exists():
    curves = pd.read_csv(path)
    for param, table in curves.groupby("param", sort=False):
        print(param)
        display(table[["value", "train PR-AUC", "val PR-AUC", "val PR-AUC std",
                       "gap PR-AUC", "val Brier"]].round(3))
        display(Image(str(RESULTS / "curves" / f"{param}.png")))
else:
    print("No curves yet: run  python -m tuning.random_forest curves")

No curves yet: run  python -m tuning.random_forest curves


### What the curves show

Each curve changes **one** setting; the others stay at the sklearn defaults with 200 trees. So the curves of settings that do not limit tree growth start from an overfit forest, around 0.64.

| Setting | Best validation PR-AUC | What the curve shows | Decision for v2 |
|---|---|---|---|
| `min_samples_leaf` | 0.714 at 10 | Plateau from 5 to 20; overfits below 5, starts to underfit at 100 | Search 5 to 30 |
| `max_depth` | 0.713 at 12 | Both ends of the bias-variance picture: depth 4 underfits (0.684, gap 0.004), unlimited overfits (0.642, gap 0.33) | Search 8 to 20 |
| `min_samples_split` | 0.714 at 100 | Same effect as the leaf size, weaker; plateau from 50 to 200 | Search 2 to 200 |
| `max_samples` | 0.685 at 30% | Fewer people per tree = more diverse trees; gap 0.33 -> 0.21 | Search 30% to 100% |
| `max_features` | 0.642 at sqrt | All 37 columns slightly worse (0.634), within the noise | Search sqrt, log2, 0.3 |
| `n_estimators` | 0.643 at 400 | Flat after 200 (0.642); the gap stays 0.33: more trees never stop memorising | Fix at 300 |
| `criterion` | 0.642 both | Gini = entropy | Fix at gini |
| `class_weight` | 0.642 at None | Weights lower PR-AUC (0.628) and damage the probabilities (Brier 0.130 -> 0.145) | Fix at None |

**Key reading.** Leaf size, depth and split size all stop at 0.713-0.714, the level of tuning v1 (0.715) and LightGBM (0.716-0.719). Different routes, same ceiling: the limit is the information in the 6 inputs, not the model.


## 4. Tuning v2

**How the search space was built.** Every choice comes from a curve in section 3: a setting with no effect or a harmful one is fixed; each searched range is the plateau of its curve plus a margin. Everything else is as in v1 (Optuna TPE sampler, seed 42, 25 trials, objective = mean 5-fold PR-AUC), so **the only difference between v1 and v2 is the search space**.

| Setting | v1 search | v2 search | Why (section 3) |
|---|---|---|---|
| `n_estimators` | 100-400 | fixed 300 | Flat after 200 trees |
| `criterion` | gini (default) | fixed gini | No difference with entropy |
| `class_weight` | None or balanced_subsample | fixed None | Weights damage the probabilities |
| `min_samples_leaf` | 1-30 | 5-30 | Plateau 5-20 |
| `max_depth` | 4-24 | 8-20 | Peak at 12; 4 underfits |
| `min_samples_split` | not searched | 2-200 | Plateau 50-200 |
| `max_samples` | not searched | 30%-100% | Smaller samples reduce overfitting |
| `max_features` | sqrt, log2, 0.5 | sqrt, log2, 0.3 | More columns slightly worse |

Run: `python -m tuning.tune --version v2 --model rf`. Optuna's first 10 trials are random search; from trial 10 its TPE sampler uses the earlier results to focus on promising regions (Session 5, 20 Oct). A full grid on these settings would need thousands of combinations.


In [ ]:
import json

# Best settings found by v1 (Soumaya) and v2 (Manel)
v1 = json.loads((ROOT / "tuning" / "results" / "v1" / "best_params.json").read_text())["Random Forest"]
v2 = json.loads((ROOT / "tuning" / "results" / "v2" / "best_params.json").read_text())["Random Forest"]
display(pd.DataFrame({"v1": pd.Series(v1), "v2": pd.Series(v2)}).astype(str))

# The four random forests, same 5 folds (from python -m tuning.random_forest baseline)
baseline = pd.read_csv(RESULTS / "baseline.csv")
forests = baseline[baseline["experiment"].str.startswith("RF")]
forests[["experiment", "train PR-AUC", "val PR-AUC", "val PR-AUC std", "gap PR-AUC",
         "val ROC-AUC", "val Brier"]].round(3)

**Reading.**
- **Same ranking as v1:** 0.715 +/- 0.013 for both. The ceiling, a third time.
- **Small gap:** 0.039 (baseline leaf 5: 0.091). The forest memorises little.
- **Trustworthy probabilities:** Brier 0.110 instead of 0.136, the level of logistic regression and LightGBM, without any recalibration. Removing the class weights the curves showed were harmful is what fixed it.
- **Every one of the 25 trials scored between 0.709 and 0.715:** the space built from the curves lies on the plateau. The random first trials already reached 0.7144; the guided ones added 0.0005. When the space is well chosen, the search method matters little.
- **The settings interact:** alone, the best leaf size was 10; with each tree seeing 39% of people, Optuna prefers 6. Two settings share the job of preventing overfitting, which is why we tune combinations, not each setting's best value.
- **Caveat:** 0.715 is the best of 25 trials picked on the same folds, so slightly optimistic. The honest number comes from the test set, once.


## 5. Findings

1. **One fully grown tree memorises** (train 0.977, validation 0.444). Bagging reduces the variance (0.444 -> 0.639) but fully grown trees still overfit (gap 0.33), and more trees do not change that.
2. **Limiting tree growth is what fixes the forest.** Leaf size, depth or split size each reach 0.713-0.714.
3. **No effect:** the splitting criterion (Gini = entropy) and the number of trees beyond 200. **Harmful:** class weights (Brier 0.130 -> 0.145).
4. **Tuning v2**, built from the curves, matches v1's ranking (0.715) with a small gap (0.039) and trustworthy probabilities (Brier 0.110 vs 0.136).
5. **The ceiling (about 0.715 PR-AUC) is the information in the 6 inputs:** three model families and three tunings land at the same level. So the bias measured in the fairness audit cannot be blamed on a badly tuned model.

**Final random forest: tuned v2** (`tuning/results/v2/best_params.json`). The team's final model remains LightGBM + Platt; the forest shows the bias-variance trade-off and confirms the ceiling. Details: findings F32 and F36 in `docs/findings_log.md`.
